<!-- # Libraries -->

In [1]:
import os, getpass, textwrap, time, uuid
from dotenv import load_dotenv # type: ignore

# from langchain_community.callbacks import get_openai_callback # type: ignore
# from langchain_core.runnables import RunnablePassthrough # type: ignore
# from langchain_core.output_parsers import StrOutputParser # type: ignore

# from langchain_groq import ChatGroq # type: ignore


In [2]:
# Load environment variables from .env file
load_dotenv()

True

In [ ]:
import nest_asyncio # type: ignore
nest_asyncio.apply()

<!-- # Helper Function -->

In [4]:
def pprint_docs(docs):
    print(f"\n{'-' * 70}\n".join([f"Document {i+1}:\n\n" + "\n".join(textwrap.wrap(d.page_content)) for i, d in enumerate(docs)]))

def pprint_result(result):
    print("Answer: " + "\n".join(textwrap.wrap(result)))

<!-- # Parsing Document to Markdown Format -->

# Convert PDF to Markdown format w Llama Parse

In [ ]:
# import os
# import logging
# from llama_cloud_services import LlamaParse

# # Setup logging
# logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
# logger = logging.getLogger(__name__)

# # Initialize parser
# parser = LlamaParse(
#     result_type="markdown",
#     parse_mode="parse_page_with_agent", 
#     preserve_layout_alignment_across_pages=True,
#     system_prompt="ini adalah dokumen pedoman Sistem Pemerintahan Berbasis Elektronik (SPBE). Didalamnya ada 4 Domain yang disingkat D (contoh: D1,D2,D3,D4), lalu ada 8 Aspek yang disingkat A (contoh: A2, A5) dan 47 Indikator yang disingkat ID (contoh: ID-3, ID-23). jadi struktur dari SPBE ada 3 unsur tadi domain, aspek, dan indikator, dan penilaian indikator ada kriteria level nya yaitu level 1-5,  dengan level 5 paling tinggi nya.\n\nakan ada perlakuan khusus saat kamu extract table pedoman SPBE ini:\n1. pada table tingkat pertama yaitu penjelasan kuesioner Domain, Aspek, dan Indikator keberapa nya, dari penjelasan, deskripsi indikator, ketentuan penilaian, dan contoh bukti dukung. ini harus dalam 1 tingkatan markdown\n2. pada table tingkat kedua yaitu penjelasan kriteria level dari suatu indikator, kriteria pemenuhan Level, kriteria bukti dukung, tolong bagi seperti itu pada tingkatan table ini",
#     language="id",
#     adaptive_long_table=True,
#     compact_markdown_table=True,
#     model="gemini-2.0-flash-001"
# )

# # Parse PDF and get markdown content
# markdown_content = parser.load_data("data/5. Pedoman Menteri PANRB NO 3 Tahun 2024 Pedoman Tata Cara Pemantauan dan Evaluasi SPBE.pdf")

# # Save markdown content to file
# output_path = "data/parsed_5. Pedoman Menteri PANRB NO 3 Tahun 2024 Pedoman Tata Cara Pemantauan dan Evaluasi SPBE.md"
# with open(output_path, "w", encoding="utf-8") as f:
#     for doc in markdown_content:
#         f.write(doc.text + "\n")

# logger.info(f"Markdown content saved to {output_path}")


<!-- # Load Markdown File -->

# Load Markdown file

In [5]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader # type: ignore
from langchain_core.documents import Document # type: ignore

markdown_path = "data/parsed_5. Pedoman Menteri PANRB NO 3 Tahun 2024 Pedoman Tata Cara Pemantauan dan Evaluasi SPBE.md" 
loader = UnstructuredMarkdownLoader(markdown_path, mode="elements")

spbe_md = loader.load()
print(f"Number of documents: {len(spbe_md)}\n")

for document in spbe_md[:50]:
    print(f"{document.page_content}\n")


Number of documents: 1826

SALINAN

MENTERI PENDAYAGUNAAN APARATUR NEGARA DAN REFORMASI BIROKRASI REPUBLIK INDONESIA

PEDOMAN MENTERI PENDAYAGUNAAN APARATUR NEGARA DAN REFORMASI BIROKRASI REPUBLIK INDONESIA NOMOR 3 TAHUN 2024 TENTANG TATA CARA PEMANTAUAN DAN EVALUASI SISTEM PEMERINTAHAN BERBASIS ELEKTRONIK

DENGAN RAHMAT TUHAN YANG MAHA ESA

MENTERI PENDAYAGUNAAN APARATUR NEGARA DAN REFORMASI BIROKRASI REPUBLIK INDONESIA,

BAB I PENDAHULUAN

A. Latar Belakang

Sistem Pemerintahan Berbasis Elektronik (SPBE) merupakan penyelenggaraan pemerintahan yang memanfaatkan teknologi informasi dan komunikasi dalam rangka meningkatkan kualitas layanan administrasi pemerintahan dan pelayanan publik yang efisien dan optimal, merupakan amanat pelaksanaan Peraturan Presiden Republik Indonesia Nomor 95 Tahun 2018 tentang Sistem Pemerintahan Berbasis Elektronik (Perpres SPBE). Pelaksanaan SPBE menjadi fondasi serta sebagai pengungkit (enabler) dari reformasi birokrasi melalui pelaksanaan transformasi dig

# Markdown Splitter

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter # type: ignore
from langchain_text_splitters import RecursiveCharacterTextSplitter # type: ignore
# menggabungkan semua teks dari `spbe_md` menjadi satu string
markdown_text = "\n".join(doc.page_content for doc in spbe_md)

headers_to_split_on = [
    ("# SALINAN", "Title 1"),
    ("# BAB", "BAB"),
    ("## A", "SubBAB A"),
    ("## B", "SubBAB B"),
    ("## C", "SubBAB C"),
    ("## D", "SubBAB D"),
    ("## E", "SubBAB E"),
    ("## F", "SubBAB F"),
    ("## G", "SubBAB G"),
    ("# Indikator", "Indikator"),
    ("## Level", "Level"), 
]

# MD splits
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on, strip_headers=False, return_each_line=True
)
md_header_splits = markdown_splitter.split_text(markdown_text)


In [ ]:
# Char-level splits
chunk_size = 1500
chunk_overlap = 100
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap
)

# Split
chunk_byMarkdown = text_splitter.split_documents(md_header_splits)
len(chunk_byMarkdown)

In [ ]:
for i, chunk in enumerate(chunk_byMarkdown):
    print(f"Chunk: {i+1}: \n{chunk.page_content}\n")

<!-- # Chunking the SPBE with Parent Document Retriever -->

<!-- Splitting Parent and Child Chunk -->

# Parent Document Splitter

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter # type: ignore

# Create a splitter for the parent documents
text_parent = RecursiveCharacterTextSplitter(chunk_size = 1500)

# Create a splitter for the child documents
# Note: child documents should be smaller than parent documents
text_child = RecursiveCharacterTextSplitter(chunk_size = 400)

<!-- Load Embedding Model -->

# Load Embedding Model

In [8]:
from langchain_ollama import OllamaEmbeddings # type: ignore

# load embedding model
embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest",
)

<!-- # Load Chroma as Vectorstore -->

# Initialize Vector Database ChromaDB

In [9]:
# Initialize a vector store to storing the chunks
from langchain_chroma import Chroma # type: ignore

parent_vstore = Chroma(
    collection_name="new2_spbe_parent_vector",
    embedding_function=embeddings,
    persist_directory ="./new2_chroma_spbe_parent_vector_db"
)

# initialize in-memory storage for the parent chunks
from langchain.storage import InMemoryStore # type: ignore
store_parent = InMemoryStore()

<!-- # Parent Document Retriever -->

# Initialize Parent Document Retriever

In [10]:
# Create a parent document retriever
from langchain.retrievers import ParentDocumentRetriever # type: ignore

parent_retriever = ParentDocumentRetriever(
    vectorstore=parent_vstore,
    docstore=store_parent, 
    child_splitter=text_child,
    parent_splitter=text_parent,
    search_kwargs={"k": 3}
)


In [11]:
# Cek jumlah dokumen
count = parent_vstore._collection.count()
print(f"Jumlah dokumen dalam ChromaDB: {count}")

Jumlah dokumen dalam ChromaDB: 2017


In [ ]:
# Hapus semua dokumen dari collection
if parent_vstore._collection:
    # Dapatkan semua IDs
    all_ids = parent_vstore._collection.get()['ids']
    if all_ids:
        parent_vstore._collection.delete(ids=all_ids)

# Cek jumlah dokumen yang tersimpan
print(f"Jumlah dokumen dalam ChromaDB: {parent_vstore._collection.count()}")

Jumlah dokumen dalam ChromaDB: 0


In [15]:
# add documents to vectorstore
parent_retriever.add_documents(md_header_splits)

In [16]:
# Cek jumlah dokumen yang tersimpan
print(f"Jumlah dokumen dalam ChromaDB: {parent_vstore._collection.count()}")

Jumlah dokumen dalam ChromaDB: 2017


<!-- # 4. Load LLM -->

# Initialize LLM from GroqCloud: Llama-3.3-70b

In [40]:
from langchain_groq import ChatGroq # type: ignore

# Verify API key is loaded
if not os.getenv("GROQ_API_KEY_RAGAS_REGIMR"):
    raise ValueError("GROQ_API_KEY not found in .env file")


model = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0,
    groq_api_key=os.getenv("GROQ_API_KEY_RAGAS_REGIMR"),
)

<!-- # 5. Routing and Prompt Template -->

# Prompt Routing

In [41]:
from langchain_core.prompts import PromptTemplate # type: ignore
from langchain_core.output_parsers import StrOutputParser # type: ignore

classification_template = PromptTemplate.from_template(
    """Kamu adalah ahli dalam menentukan jenis pertanyaan terkait SPBE.
    akan diberikan pertanyaan dari pengguna dibawah, tolong bantu klasifikasi apakah pertanyaan terkait SPBE `SPBE Umum` atau `SPBE Audit`.
    <jika pertanyaan adalah terkait hal umum yang ditanyakan terkait Sistem Pemerintahan Berbasis Elektronik (SPBE) seperti definisi, latar belakang, indeks tahun 2021-2023, dan keterangan alasan lainnya, maka klasifikasi pertanyaan sebagai 'SPBE Umum'>
    <jika pertanyaan sudah terkait studi kasus, memberikan permasalahan terkait penilaian indeks SPBE, kriteria, bukti dukung, indikator penilaian, tingkat kematangan berapa dan aspek lainnya yang kamu bisa tahu ini adalah pekerjaan audit dan juga jika opsi yang diminta untuk menentukan level itu termasuk kegiatan audit, maka klasifikasi pertanyaan sebagai 'SPBE Audit'>
    jangan berikan alasan apapun, cukup tolong klasifikasi saja jawabannya,
    - contoh pertanyaan:
    1. Apa saja domain yang ada di SPBE?
    - harapan jawaban klasifikasi: SPBE Umum
    2. Berapa indeks SPBE nasional tahun 2021-2023?
    - harapan jawaban klasifikasi: SPBE Umum
    3. Pada Indikator 32, Kab. Katingan melampirkan bukti dukung berupa screenshoot aplikasi SIPD, berapakah tingkat kematangan yang dapat diberikan dengan bukti dukung tersebut? Daftar Dokumen Pendukung: F2101-543-Indikator_32
    - harapan jawaban klasifikasi: SPBE Audit
    4. Penerapan Manajemen hanya dapat mencapai tingkat kematangan level 1 dikarenakan tidak?
    - harapan jawaban klasifikasi: SPBE Umum

    <pertanyaan>
    {question}
    </pertanyaan>

    Klasifikasi:"""
)

classification_chain = classification_template | model | StrOutputParser()

## Template Prompt

In [42]:
SPBE_UMUM_TEMPLATE = """Anda adalah asisten asesor SPBE yang detail, khususnya dalam memberikan informasi Sistem Pemerintahan Berbasis Elektronik (SPBE) Indonesia.

terkait Sistem Pemerintahan Berbasis Elektronik (SPBE). Didalamnya ada 4 Domain yang disingkat D (contoh: D1,D2,D3,D4), lalu ada 8 Aspek yang disingkat A (contoh: A2, A5) dan 47 Indikator yang disingkat ID (contoh: ID-3, ID-23). jadi struktur dari SPBE ada 3 unsur tadi domain, aspek, dan indikator, dan penilaian indikator ada kriteria level nya yaitu level 1-5,  dengan level 5 paling tinggi nya.
jika ditanya terkait Indeks SPBE Nasional, tolong cari dan referensikan ke Tabel 1. Indeks SPBE Nasional (2021 - 2023)

Ada 4 Domain utama SPBE:
1. Domain 1: Kebijakan
2. Domain 2: Tata Kelola
3. Domain 3: Manajemen
4. Domain 4: Layanan

Ada 8 Aspek utama SPBE:
1. Aspek 1: Kebijakan Tata Kelola SPBE
2. Aspek 2: Perencanaan Strategis SPBE
3. Aspek 3: Teknologi Informasi dan Komunikasi
4. Aspek 4: Penyelenggaraan SPBE
5. Aspek 5: Penerapan Manajemen SPBE
6. Aspek 6: Audit TIK
7. Aspek 7: Layanan Administrasi Pemerintahan
8. Aspek 8: Layanan Publik

Ada 47 Indikator utama SPBE:
1. Indikator 1-10 berada di Domain Kebijakan (bobot 13%)
2. Indikator 11-20 berada di Domain Tata Kelola (bobot 25%)
3. Indikator 21-31 berada di Domain Manajemen (bobot 16,5%)
4. Indikator 32-47 berada di Domain Layanan (bobot 45,5%)

Ada 5 Tingkat Kematangan Domain Kebijakan, Tata Kelola, dan Manajemen:
1. Rintisan
2. Terkelola
3. Terdefinisi
4. Terpadu dan Terukur
5. Optimum

Sedangkan untuk Domain Layanan:
1. Informasi
2. Interaksi
3. Transaksi
4. Kolaborasi
5. Optimum

## 🎯 Aturan Menjawab dan Tugas:
1. Menjawab pertanyaan umum tentang SPBE secara akurat.
2. Menunjukkan domain, aspek, dan indikator juga level terkait dengan pertanyaan pengguna berdasarkan pedoman SPBE jika pengguna menanyakan indikator/aspek/domain dan level nya juga memberikan alasan pemberian penilaian dari deskripsi ataupun kriteria, bukti lainnya.
3. Sebelum sebuah tingkat kematangan berada pada suatu level, harus memenuhi semua kriteria dan bukti dukung level sebelumnya kecuali level 1 yang masih awal.

## 📢 Tanggapan Anda harus mengikuti aturan berikut:
- Jika pengguna hanya ingin ringkasan, berikan jawaban singkat.
- jawab dengan konkrit tidak terlalu panjang dan jelas.
- Jika pertanyaan tidak terkait dengan SPBE dan pedoman SPBE, berikan respons berikut:
  - *"Maaf, pertanyaan tersebut tidak terkait dengan SPBE KemenpanRB. Saya hanya dapat memberikan jawaban berdasarkan konteks tersebut."
---

## 🔍 Konteks Riwayat Percakapan:
{chat_history}

## 🔍 Konteks yang diberikan dari dokumen:
{context}

## ❓ Pertanyaan pengguna:
{question}

## ✅ Jawaban Anda:
"""

SPBE_Audit_Template = """
Anda adalah asisten asesor SPBE yang detail, khususnya dalam membantu auditing Sistem Pemerintahan Berbasis Elektronik (SPBE) Indonesia.

terkait Sistem Pemerintahan Berbasis Elektronik (SPBE). Didalamnya ada 4 Domain yang disingkat D (contoh: D1,D2,D3,D4), lalu ada 8 Aspek yang disingkat A (contoh: A2, A5) dan 47 Indikator yang disingkat ID (contoh: ID-3, ID-23). jadi struktur dari SPBE ada 3 unsur tadi domain, aspek, dan indikator, dan penilaian indikator ada kriteria level nya yaitu level 1-5,  dengan level 5 paling tinggi nya.
jika ditanya terkait Indeks SPBE Nasional, tolong cari dan referensikan ke Tabel 1. Indeks SPBE Nasional (2021 - 2023)

Ada 4 Domain utama SPBE:
1. Domain 1: Kebijakan
2. Domain 2: Tata Kelola
3. Domain 3: Manajemen
4. Domain 4: Layanan

Ada 8 Aspek utama SPBE:
1. Aspek 1: Kebijakan Tata Kelola SPBE
2. Aspek 2: Perencanaan Strategis SPBE
3. Aspek 3: Teknologi Informasi dan Komunikasi
4. Aspek 4: Penyelenggaraan SPBE
5. Aspek 5: Penerapan Manajemen SPBE
6. Aspek 6: Audit TIK
7. Aspek 7: Layanan Administrasi Pemerintahan
8. Aspek 8: Layanan Publik

Ada 47 Indikator utama SPBE:
1. Indikator 1-10 berada di Domain Kebijakan (bobot 13%)
2. Indikator 11-20 berada di Domain Tata Kelola (bobot 25%)
3. Indikator 21-31 berada di Domain Manajemen (bobot 16,5%)
4. Indikator 32-47 berada di Domain Layanan (bobot 45,5%)

Ada 5 Tingkat Kematangan Domain Kebijakan, Tata Kelola, dan Manajemen:
1. Rintisan
2. Terkelola
3. Terdefinisi
4. Terpadu dan Terukur
5. Optimum

Sedangkan untuk Domain Layanan:
1. Informasi
2. Interaksi
3. Transaksi
4. Kolaborasi
5. Optimum

## 🎯 Aturan Menjawab dan Tugas:
1. Dalam memberikan penilaian level berapa sebuah indikator, maka pastikan memenuhi kriteria pemenuhan level dan kriteria bukti dukung, ini harus ada jika melakukan audit, kalau tidak ada maka mintakan kepada pengguna hal tersebut.
2. Menunjukkan domain, aspek, dan indikator juga level terkait dengan pertanyaan pengguna berdasarkan pedoman SPBE jika pengguna menanyakan indikantor/aspek/doman dan level nya juga memberikan alasan pemberian penilaian dari deskripsi ataupun kriteria, bukti lainnya.
3. Sebelum sebuah tingkat kematangan berada pada suatu level, harus memenuhi semua kriteria dan bukti dukung level sebelumnya kecuali level 1 yang masih awal.
4. Kamu akan diberikan konteks dibawah, jadi pastikan hal yang dibutuhkan untuk penilaian tersedia, jangan menjawab jika konteks tidak tersedia

## 📢 Tanggapan Anda harus mengikuti aturan berikut:
- jawab dengan konkrit tidak terlalu panjang dan jelas.
- Jika pertanyaan tidak terkait dengan SPBE dan pedoman SPBE, berikan respons berikut:
  - *"Maaf, pertanyaan tersebut tidak terkait dengan SPBE KemenpanRB. Saya hanya dapat memberikan jawaban berdasarkan konteks tersebut."
---
Berikut contoh format jawaban yang harus kamu lakukan:
<contoh format jawaban>
Berdasarkan pedoman pemantauan dan evaluasi SPBE.....dengan diberikannya informasi tahapan, kriteria pemenuhan level dan bukti dukung maka indeks SPBE diberikan: 

{{ 
  "Penilaian": {{
    "Level": 5,
    "Indikator": 3,
    "Aspek": 2,
    "Domain": 1
  }}
}}

alasan penilaian diatas karena.....level.....indikator....aspek....domain...sesuai standard dari pedoman evaluasi SPBE.
</contoh format jawaban>
yang titik diatas kamu harus isi alasannya, dan sesuai format

## 🔍 Konteks Riwayat Percakapan:
{chat_history}

## 🔍 Konteks yang diberikan dari dokumen:
{context}

## ❓ Pertanyaan pengguna:
{question}

## ✅ Jawaban Anda:
"""

In [43]:
from langchain_core.runnables import RunnableLambda # type: ignore

def get_classification_prompt(input_query):
    return classification_chain.invoke({"question": input_query["question"]}).strip()

def prompt_router(input_query):
    classification_query = classification_chain.invoke({"question": input_query["question"]}).strip()

    if classification_query == "SPBE Umum":
        prompt = PromptTemplate(template=SPBE_UMUM_TEMPLATE, input_variables=["chat_history", "context", "question"])
    elif classification_query == "SPBE Audit":
        prompt = PromptTemplate(template=SPBE_Audit_Template, input_variables=["chat_history", "context", "question"])
    else:
        print("Unexpected classification: ", classification_query)
        
    return prompt

<!-- # Function Invoke -->

In [ ]:
from langchain.memory import ConversationBufferMemory # type: ignore
from langchain.chains import ConversationalRetrievalChain # type: ignore
# Create memory with explicit chat history
memory2 = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key='answer'
)

# Chain Parent Retriever

In [ ]:
# from langchain.chains import ConversationalRetrievalChain

# # Create chain with explicit source document handling
# parent_chain = ConversationalRetrievalChain.from_llm(
#     llm=model,
#     retriever=parent_retriever,
#     memory=memory,
#     return_source_documents=True,
#     combine_docs_chain_kwargs={
#         "prompt": prompt,
#         "document_variable_name": "context"
#     },
#     verbose=True  # Enable verbose mode for debugging
# )

# chat_history = []
# result = parent_chain.invoke({
#     "chat_history": chat_history,
#     "question": question
# })
# response = result["answer"]
# print(response)
# source_documents = result.get("source_documents", [])

In [ ]:
# pprint_docs(parent_retriever.invoke(questions[0]))

# MultiQuery Combine to Parent Document Retriever

In [45]:
from langchain.retrievers.multi_query import MultiQueryRetriever # type: ignore

multi_parent_retriever = MultiQueryRetriever.from_llm(
    retriever=parent_retriever, llm=model
)

In [46]:
# Set logging for the queries
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

question_multi_query = """Dalam penerapan Manajemen Risiko SPBE, masing-masing IPPD (Instansi Pusat dan Pemerintah Daerah) dapat mengacu pada Pedoman Manajemen Risiko SPBE yang ditetapkan dalam kebijakan:"""

unique_docs = multi_parent_retriever.invoke(question_multi_query)
len(unique_docs)

# Getting Classification Prompt
classification_prompt_result = get_classification_prompt({"question": question_multi_query})
print(f"🔍 Hasil Klasifikasi: {classification_prompt_result}")

INFO:langchain.retrievers.multi_query:Generated queries: ['Berikut adalah tiga versi alternatif dari pertanyaan asli untuk membantu mengatasi keterbatasan pencarian kesamaan berbasis jarak:', 'Apa pedoman yang digunakan oleh Instansi Pusat dan Pemerintah Daerah dalam menerapkan Manajemen Risiko SPBE?', 'Dalam konteks Manajemen Risiko SPBE, apa kebijakan yang menjadi acuan bagi IPPD (Instansi Pusat dan Pemerintah Daerah) untuk mengelola risiko?', 'Bagaimana IPPD (Instansi Pusat dan Pemerintah Daerah) dapat menerapkan Manajemen Risiko SPBE sesuai dengan pedoman yang telah ditetapkan dalam kebijakan?']


🔍 Hasil Klasifikasi: SPBE Umum


In [47]:
def run_multi_parent_chain(question):
    """
    Function to run multi-parent chain with error handling
    """
    try:
        prompt = prompt_router({"question": question})
        multi_parent_chain = ConversationalRetrievalChain.from_llm(
            llm=model,
            retriever=multi_parent_retriever,
            return_source_documents=True,
            combine_docs_chain_kwargs={
                "prompt": prompt,
                "document_variable_name": "context"
            },
            verbose=True  # Enable verbose mode for debugging
        )

        chat_history10 = []
        result10 = multi_parent_chain.invoke({
            "chat_history": chat_history10,
            "question": question
        })
        print(result10["answer"])
        return result10
        
    except Exception as e:
        print(f"Error occurred: {str(e)}")
        return None

# Call the function to execute and display results
# run_multi_parent_chain(question_multi_query)

In [ ]:
# chat_history10 = []
# result10 = multi_parent_chain.invoke({
#     "chat_history": chat_history10,
#     "question": question_multi_query
# })
# response10 = result10["answer"]
# print(response10)
# # source_documents = result.get("source_documents", [])

# Evaluasi RAG with RAGAS

In [48]:
def prepare_ragas_data(result, references=None):
    """
    Menyiapkan data untuk evaluasi RAGAS dari result function run_multi_parent_chain
    """
    if result is None:
        return None 
    
    # Mengakses answer dan context
    generated_answer = result["answer"]
    source_documents = result.get("source_documents", [])
    
    # Menggabungkan context dari semua source documents
    retrieved_contexts = "\n\n".join([doc.page_content for doc in source_documents])
    
    return {
        "response": generated_answer,
        "retrieved_contexts": retrieved_contexts,
        "reference": references,
        "source_documents": source_documents
    }

In [49]:
import time

def evaluate_with_ragas(questions, references=None):
    """
    Evaluasi menggunakan RAGAS dengan function run_multi_parent_chain yang sudah ada
    """
    ragas_dataset = []
    
    for i, question in enumerate(questions):
        print(f"Processing question {i+1}/{len(questions)}: {question[:50]}...")
        

        result = run_multi_parent_chain(question)
        
        if result:
            # Mendapatkan ground truth jika ada
            reference = references[i] if references and i < len(references) else None
            
            # Menyiapkan data RAGAS
            ragas_data = prepare_ragas_data(result, reference)
            
            if ragas_data:
                # Menambahkan question ke data
                ragas_data["user_input"] = question
                ragas_dataset.append(ragas_data)
                print(f"✅ Question {i+1} processed successfully")
            else:
                print(f"❌ Failed to prepare RAGAS data for question {i+1}")
        else:
            print(f"❌ Failed to process question {i+1}")

        # Jeda
        print("⏳ Menunggu 30 detik...")
        time.sleep(30)
    
    return ragas_dataset

In [50]:
# Contoh penggunaan
questions_for_evaluation = [
    """Dalam penerapan Manajemen Risiko SPBE, masing-masing IPPD (Instansi Pusat dan Pemerintah Daerah) dapat mengacu pada Pedoman Manajemen Risiko SPBE yang ditetapkan dalam kebijakan:
""",
    """Jika "Rencana dan Anggaran SPBE Instansi Pusat Pemerintah Daerah telah terpadu dan dapat dikendalikan oleh unit kerja/perangkat daerah yang menjalankan fungsi perencanaan dan penganggaran dan telah direviu serta dievaluasi secara periodik:" Maka dapat diberikan level...
    """,
    "Berapa nilai indeks domain yang masih dibawah target pada tahun 2021-2023?",
    """Sebuah IPPD melampirkan Peta Rencana SPBE yang telah didokumentasikan secara formal, dan mengklaim memiliki dokumen Peta Rencana SPBE yang telah mengatur seluruh muatan Peta Rencana SPBE Instansi Pusat Pemerintah Daerah. Dokumen Peta Rencana yang diunggah berisikan muatan Peta Rencana secara lengkap antara lain Tata Kelola SPBE, Manajemen SPBE, Layanan SPBE, Arsitektur SPBE, Aplikasi SPBE, Keamanan SPBE dan Audit TIK. Maka level yang pantas diberikan adalah...
""",
    """Dibawah ini merupakan domain-domain dari Arsitektur SPBE berdasarkan Perpres SPBE, kecuali (pilih salah satu):

- domain arsitektur Proses Bisnis
- domain arsitektur Manajemen SPBE 
- domain arsitektur Infrastruktur SPBE
- domain arsitektur Aplikasi SPBE
- domain arsitektur Keamanan SPBE
- domain arsitektur Layanan SPBE
"""
]

references = [
    """PermenPANRB No. 5 Tahun 2020 memberikan pedoman umum bagi Instansi Pusat dan Pemerintah Daerah dalam melaksanakan SPBE, termasuk penerapan Manajemen Risiko SPBE
    """,
    """Berdasarkan pedoman pemantauan dan evaluasi SPBE, dengan diberikannya informasi kriteria-kriteria, maka indeks SPBE yang diberikan: 
 
Penilaian: 
Level: 4,
Indikator: 13,
Aspek: 2,
Domain: 2

Alasan penilaian diatas karena Rencana dan Anggaran SPBE Instansi Pusat/Pemerintah Daerah telah terpadu dan dapat dikendalikan oleh unit kerja/perangkat daerah yang menjalankan fungsi perencanaan dan penganggaran dan telah direviu serta dievaluasi secara periodik, sehingga sesuai dengan kriteria Level 4 pada Domain Tata Kelola, Aspek Perencanaan Strategis SPBE, dan Indikator 13 tentang tingkat kematangan keterpaduan rencana dan anggaran SPBE.
""",
    """Nilai indeks domain yang masih di bawah target (<2,60) pada tahun 2021-2023 adalah:

1. Indeks Domain Tata Kelola: 
   - Tahun 2021: 1,89
   - Tahun 2022: 1,85
   - Tahun 2023: 2,29

2. Indeks Domain Manajemen: 
   - Tahun 2021: 1,23
   - Tahun 2022: 1,32
   - Tahun 2023: 1,66
   """,
    """Berdasarkan pedoman pemantauan dan evaluasi SPBE, dengan diberikannya informasi kriteria-kriteria, maka indeks SPBE yang diberikan: 
 
Penilaian: 
Level: 3,
Indikator: 12,
Aspek: 2,
Domain: 2

Alasan penilaian diatas karena dokumen Peta Rencana SPBE telah mengatur seluruh muatan Peta Rencana SPBE Instansi Pusat/Pemerintah Daerah secara lengkap (Tata Kelola SPBE, Manajemen SPBE, Layanan SPBE, Infrastruktur SPBE, Aplikasi SPBE, Keamanan SPBE, Audit Teknologi SPBE dan Audit TIK) dan dokumen Peta Rencana SPBE telah didokumentasikan secara formal, sehingga sesuai dengan kriteria Level 3 pada Domain Tata Kelola, Aspek Perencanaan Strategis SPBE, dan Indikator 12 tentang Tingkat Kematangan Peta Rencana SPBE Instansi Pusat/Pemerintah Daerah. Namun, perlu diperhatikan bahwa untuk mencapai Level 4, IPPD harus memenuhi kriteria tambahan, yaitu dokumen Peta Rencana SPBE telah diterapkan secara konsisten melalui rencana kerja dan anggaran 3 (tiga) tahun terakhir, dan dokumen Peta Rencana SPBE telah dilakukan reviu dan evaluasi secara periodik.
""",
    "Domain Arsitektur Manajemen SPBE tidak termasuk dalam daftar domain arsitektur SPBE yang ditetapkan dalam Perpres SPBE. Domain Manajemen SPBE sebenarnya merupakan salah satu aspek dalam Sistem Pemerintahan Berbasis Elektronik (SPBE), bukan domain arsitektur SPBE."
]

# Menjalankan evaluasi
ragas_dataset = evaluate_with_ragas(questions_for_evaluation, references)

# Menampilkan hasil
print(f"\n📊 Dataset RAGAS berhasil disiapkan dengan {len(ragas_dataset)} sampel")
for i, data in enumerate(ragas_dataset):
    print(f"\n--- Sampel {i+1} ---")
    print(f"Question: {data['user_input']}")
    print(f"Context Length: {len(data['retrieved_contexts'])} characters")
    print(f"Response: {data['response'][:100]}...")
    if data['reference']:
        print(f"Reference: {data['reference'][:100]}...")

Processing question 1/5: Dalam penerapan Manajemen Risiko SPBE, masing-masi...


INFO:langchain.retrievers.multi_query:Generated queries: ['Apa saja pedoman manajemen risiko SPBE yang perlu diikuti oleh Instansi Pusat dan Pemerintah Daerah dalam penerapan Manajemen Risiko SPBE?', 'Dalam konteks kebijakan, bagaimana pedoman manajemen risiko SPBE dapat membantu IPPD (Instansi Pusat dan Pemerintah Daerah) mengelola risiko yang terkait dengan Sistem Pengelolaan Barang Milik Negara atau Daerah?', 'Bagaimana IPPD (Instansi Pusat dan Pemerintah Daerah) dapat menerapkan pedoman manajemen risiko SPBE yang efektif untuk mengoptimalkan pengelolaan barang milik negara atau daerah dan meminimalkan risiko yang terkait?']




> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Anda adalah asisten asesor SPBE yang detail, khususnya dalam memberikan informasi Sistem Pemerintahan Berbasis Elektronik (SPBE) Indonesia.

terkait Sistem Pemerintahan Berbasis Elektronik (SPBE). Didalamnya ada 4 Domain yang disingkat D (contoh: D1,D2,D3,D4), lalu ada 8 Aspek yang disingkat A (contoh: A2, A5) dan 47 Indikator yang disingkat ID (contoh: ID-3, ID-23). jadi struktur dari SPBE ada 3 unsur tadi domain, aspek, dan indikator, dan penilaian indikator ada kriteria level nya yaitu level 1-5,  dengan level 5 paling tinggi nya.
jika ditanya terkait Indeks SPBE Nasional, tolong cari dan referensikan ke Tabel 1. Indeks SPBE Nasional (2021 - 2023)

Ada 4 Domain utama SPBE:
1. Domain 1: Kebijakan
2. Domain 2: Tata Kelola
3. Domain 3: Manajemen
4. Domain 4: Layanan

Ada 8 Aspek utama SPBE:
1. Aspek 1: Kebijakan Tata Kelola SPBE
2. Aspek 2: Perencanaan Strategis SPBE
3. Aspek 3: Te

INFO:langchain.retrievers.multi_query:Generated queries: ['Berikut adalah tiga versi alternatif dari pertanyaan asli untuk membantu mengatasi keterbatasan pencarian kesamaan berbasis jarak:', 'Apa tingkat integrasi rencana dan anggaran SPBE di instansi pemerintah daerah yang telah mencapai tingkat kematangan dalam pengelolaan perencanaan dan penganggaran, serta melakukan review dan evaluasi secara berkala?', 'Jika suatu instansi pemerintah daerah telah berhasil mengintegrasikan rencana dan anggaran SPBE, serta melakukan pengendalian dan evaluasi secara terstruktur oleh unit kerja yang relevan, maka apa level kematangan yang dapat diberikan?', 'Bagaimana tingkat kematangan suatu instansi pemerintah daerah dalam mengelola rencana dan anggaran SPBE yang telah terintegrasi, dapat dikendalikan oleh unit kerja yang tepat, dan telah melalui proses review serta evaluasi secara periodik, sehingga layak diberikan level tertentu?']




> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Anda adalah asisten asesor SPBE yang detail, khususnya dalam membantu auditing Sistem Pemerintahan Berbasis Elektronik (SPBE) Indonesia.

terkait Sistem Pemerintahan Berbasis Elektronik (SPBE). Didalamnya ada 4 Domain yang disingkat D (contoh: D1,D2,D3,D4), lalu ada 8 Aspek yang disingkat A (contoh: A2, A5) dan 47 Indikator yang disingkat ID (contoh: ID-3, ID-23). jadi struktur dari SPBE ada 3 unsur tadi domain, aspek, dan indikator, dan penilaian indikator ada kriteria level nya yaitu level 1-5,  dengan level 5 paling tinggi nya.
jika ditanya terkait Indeks SPBE Nasional, tolong cari dan referensikan ke Tabel 1. Indeks SPBE Nasional (2021 - 2023)

Ada 4 Domain utama SPBE:
1. Domain 1: Kebijakan
2. Domain 2: Tata Kelola
3. Domain 3: Manajemen
4. Domain 4: Layanan

Ada 8 Aspek utama SPBE:
1. Aspek 1: Kebijakan Tata Kelola SPBE
2. Aspek 2: Perencanaan Strategis SPBE
3. Aspek 3: Tekn

...



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Anda adalah asisten asesor SPBE yang detail, khususnya dalam memberikan informasi Sistem Pemerintahan Berbasis Elektronik (SPBE) Indonesia.

terkait Sistem Pemerintahan Berbasis Elektronik (SPBE). Didalamnya ada 4 Domain yang disingkat D (contoh: D1,D2,D3,D4), lalu ada 8 Aspek yang disingkat A (contoh: A2, A5) dan 47 Indikator yang disingkat ID (contoh: ID-3, ID-23). jadi struktur dari SPBE ada 3 unsur tadi domain, aspek, dan indikator, dan penilaian indikator ada kriteria level nya yaitu level 1-5,  dengan level 5 paling tinggi nya.
jika ditanya terkait Indeks SPBE Nasional, tolong cari dan referensikan ke Tabel 1. Indeks SPBE Nasional (2021 - 2023)

Ada 4 Domain utama SPBE:
1. Domain 1: Kebijakan
2. Domain 2: Tata Kelola
3. Domain 3: Manajemen
4. Domain 4: Layanan

Ada 8 Aspek utama SPBE:
1. Aspek 1: Kebijakan Tata Kelola SPBE
2. Aspek 2: Perencanaan Strategis SPBE
3. Aspek 3: Te

INFO:langchain.retrievers.multi_query:Generated queries: ['Berikut adalah tiga versi alternatif dari pertanyaan asli untuk membantu mengatasi keterbatasan pencarian kesamaan berbasis jarak:', '1. Apa tingkat yang sesuai untuk sebuah IPPD yang telah melampirkan Peta Rencana SPBE yang lengkap dan formal, mencakup aspek-aspek seperti Tata Kelola, Manajemen, Layanan, Arsitektur, Aplikasi, Keamanan, dan Audit TIK?', '2. Bagaimana menentukan level yang tepat untuk sebuah Instansi Pusat Pemerintah Daerah yang telah mengembangkan dan mendokumentasikan Peta Rencana SPBE secara menyeluruh, termasuk seluruh komponen yang diperlukan seperti Tata Kelola SPBE, Manajemen SPBE, Layanan SPBE, Arsitektur SPBE, Aplikasi SPBE, Keamanan SPBE, dan Audit TIK?', '3. Berdasarkan informasi yang diberikan tentang sebuah IPPD yang telah menyusun Peta Rencana SPBE yang komprehensif, mencakup berbagai aspek seperti Tata Kelola, Manajemen, Layanan, Arsitektur, Aplikasi, Keamanan, dan Audit TIK, apa level yang paling



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Anda adalah asisten asesor SPBE yang detail, khususnya dalam membantu auditing Sistem Pemerintahan Berbasis Elektronik (SPBE) Indonesia.

terkait Sistem Pemerintahan Berbasis Elektronik (SPBE). Didalamnya ada 4 Domain yang disingkat D (contoh: D1,D2,D3,D4), lalu ada 8 Aspek yang disingkat A (contoh: A2, A5) dan 47 Indikator yang disingkat ID (contoh: ID-3, ID-23). jadi struktur dari SPBE ada 3 unsur tadi domain, aspek, dan indikator, dan penilaian indikator ada kriteria level nya yaitu level 1-5,  dengan level 5 paling tinggi nya.
jika ditanya terkait Indeks SPBE Nasional, tolong cari dan referensikan ke Tabel 1. Indeks SPBE Nasional (2021 - 2023)

Ada 4 Domain utama SPBE:
1. Domain 1: Kebijakan
2. Domain 2: Tata Kelola
3. Domain 3: Manajemen
4. Domain 4: Layanan

Ada 8 Aspek utama SPBE:
1. Aspek 1: Kebijakan Tata Kelola SPBE
2. Aspek 2: Perencanaan Strategis SPBE
3. Aspek 3: Tekn

INFO:langchain.retrievers.multi_query:Generated queries: ['Apa saja domain arsitektur SPBE yang tercantum dalam Perpres SPBE, kecuali satu yang tidak termasuk?', ' ', 'Berikut ini adalah daftar domain arsitektur SPBE berdasarkan Perpres SPBE, kecuali salah satu, pilih yang tidak termasuk: domain arsitektur Proses Bisnis, domain arsitektur Manajemen SPBE, domain arsitektur Infrastruktur SPBE, domain arsitektur Aplikasi SPBE, domain arsitektur Keamanan SPBE, domain arsitektur Layanan SPBE, mana yang bukan termasuk?', ' ', 'Domain-domain apa saja yang termasuk dalam arsitektur SPBE menurut Perpres SPBE, dan mana yang tidak termasuk dalam daftar: domain arsitektur Proses Bisnis, domain arsitektur Manajemen SPBE, domain arsitektur Infrastruktur SPBE, domain arsitektur Aplikasi SPBE, domain arsitektur Keamanan SPBE, domain arsitektur Layanan SPBE?']




> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Anda adalah asisten asesor SPBE yang detail, khususnya dalam memberikan informasi Sistem Pemerintahan Berbasis Elektronik (SPBE) Indonesia.

terkait Sistem Pemerintahan Berbasis Elektronik (SPBE). Didalamnya ada 4 Domain yang disingkat D (contoh: D1,D2,D3,D4), lalu ada 8 Aspek yang disingkat A (contoh: A2, A5) dan 47 Indikator yang disingkat ID (contoh: ID-3, ID-23). jadi struktur dari SPBE ada 3 unsur tadi domain, aspek, dan indikator, dan penilaian indikator ada kriteria level nya yaitu level 1-5,  dengan level 5 paling tinggi nya.
jika ditanya terkait Indeks SPBE Nasional, tolong cari dan referensikan ke Tabel 1. Indeks SPBE Nasional (2021 - 2023)

Ada 4 Domain utama SPBE:
1. Domain 1: Kebijakan
2. Domain 2: Tata Kelola
3. Domain 3: Manajemen
4. Domain 4: Layanan

Ada 8 Aspek utama SPBE:
1. Aspek 1: Kebijakan Tata Kelola SPBE
2. Aspek 2: Perencanaan Strategis SPBE
3. Aspek 3: Te

In [51]:
print(ragas_dataset)

[{'response': 'Dalam penerapan Manajemen Risiko SPBE, masing-masing IPPD (Instansi Pusat dan Pemerintah Daerah) dapat mengacu pada Pedoman Manajemen Risiko SPBE yang ditetapkan dalam PermenPANRB No. 5/2020.', 'retrieved_contexts': 'INDIKATOR 21\nDomain Aspek Ind Kuesioner D3 A5 ID-21 Tingkat Kematangan Penerapan Manajemen Risiko SPBE. Deskripsi Indikator: a. Manajemen Risiko SPBE adalah pendekatan sistematis yang meliputi proses, pengukuran, struktur, dan budaya untuk menentukan tindakan terbaik terkait Risiko SPBE; b. Risiko SPBE adalah peluang terjadinya suatu peristiwa yang akan mempengaruhi keberhasilan terhadap pencapaian tujuan penerapan SPBE; c. Manajemen Risiko bertujuan untuk menjamin keberlangsungan SPBE dengan meminimalkan dampak risiko dalam SPBE; d. Instansi Pusat dan Pemerintah Daerah menerapkan manajemen risiko SPBE berdasarkan pedoman Manajemen Risiko SPBE. Ketentuan Penilaian: Penilaian dilakukan terhadap bukti dukung penerapan manajemen risiko Instansi Pusat/Pemerinta

In [ ]:
# Konversi ke DataFrame
import pandas as pd # type: ignore
df = pd.DataFrame(ragas_dataset)
df

,response,retrieved_contexts,reference,source_documents,user_input
0,"Dalam penerapan Manajemen Risiko SPBE, masing-...",INDIKATOR 21\nDomain Aspek Ind Kuesioner D3 A5...,PermenPANRB No. 5 Tahun 2020 memberikan pedoma...,[page_content='INDIKATOR 21\nDomain Aspek Ind ...,"Dalam penerapan Manajemen Risiko SPBE, masing-..."
1,Berdasarkan pedoman pemantauan dan evaluasi SP...,d. Tidak menerima gratifikasi terkait dengan p...,Berdasarkan pedoman pemantauan dan evaluasi SP...,[page_content='d. Tidak menerima gratifikasi t...,"Jika ""Rencana dan Anggaran SPBE Instansi Pusat..."
2,Nilai indeks domain yang masih di bawah target...,Berdasarkan capaian Indeks SPBE Nasional sebag...,Nilai indeks domain yang masih di bawah target...,[page_content='Berdasarkan capaian Indeks SPBE...,Berapa nilai indeks domain yang masih dibawah ...
3,Berdasarkan pedoman pemantauan dan evaluasi SP...,d. Tidak menerima gratifikasi terkait dengan p...,Berdasarkan pedoman pemantauan dan evaluasi SP...,[page_content='d. Tidak menerima gratifikasi t...,Sebuah IPPD melampirkan Peta Rencana SPBE yang...
4,Domain-domain dari Arsitektur SPBE berdasarkan...,jdih.menpan.go.id\n50-\nd. Arsitektur SPBE Pem...,Domain Arsitektur Manajemen SPBE tidak termasu...,[page_content='jdih.menpan.go.id\n50-\nd. Arsi...,Dibawah ini merupakan domain-domain dari Arsit...


In [53]:
# Buat list baru dengan urutan dan filter kolom yang diinginkan
ragas_dataset_filtered = [
    {
        "user_input": item["user_input"],
        "retrieved_contexts": item["retrieved_contexts"],
        "response": item["response"],
        "reference": item["reference"]
    }
    for item in ragas_dataset
]

for item in ragas_dataset_filtered:
    if isinstance(item["retrieved_contexts"], str):
        item["retrieved_contexts"] = item["retrieved_contexts"].split("\n\n")

In [54]:
df_ragas = pd.DataFrame(ragas_dataset_filtered, columns=["user_input", "retrieved_contexts", "response", "reference"])
df_ragas.to_csv("spbe2_test_vector_ragas.csv")

In [55]:
import pickle
with open("vector2_ragas_dataset_filtered.pkl", "wb") as f:
    pickle.dump(ragas_dataset_filtered, f)

In [56]:
import pickle
with open("vector2_ragas_dataset_filtered.pkl", "rb") as f:
    ragas_dataset_filtered = pickle.load(f)

In [ ]:
# Load environment variables from .env file
load_dotenv()
import pandas as pd # type: ignore
from langchain_groq import ChatGroq # type: ignore
from ragas import evaluate, EvaluationDataset # type: ignore
from ragas.run_config import RunConfig # type: ignore
from ragas.llms import LangchainLLMWrapper # type: ignore
from ragas.embeddings import LangchainEmbeddingsWrapper # type: ignore
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings # type: ignore
from ragas.metrics import ( # type: ignore
    faithfulness,  
    answer_relevancy, 
    context_recall, 
    context_precision
    ) 

eval_dataset = EvaluationDataset.from_list(ragas_dataset_filtered)

embeddings_ollama = OllamaEmbeddings(
    model="nomic-embed-text:latest",
)

# Integration Azure Open AI in Langchain
llm_openai_azure = AzureChatOpenAI(
    openai_api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    deployment_name=os.environ.get("41_AZURE_OPENAI_DEPLOYMENT"),
    openai_api_version=os.environ.get("41_AZURE_OPENAI_API_VERSION"),
)

# llm2_groqcloud = ChatGroq(
#     model_name="llama-3.3-70b-versatile",
#     temperature=0,
#     groq_api_key=os.getenv("GROQ_API_KEY_RAGAS_REGIMR"),
# )

evaluator_embedding = LangchainEmbeddingsWrapper(embeddings_ollama)
evaluator_llm = LangchainLLMWrapper(llm_openai_azure)

# Evaluasi dengan RAGAS
results = evaluate(
    dataset=eval_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_recall,
        context_precision
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embedding,
)

print("📊 Hasil Evaluasi RAGAS:")
results_df = results.to_pandas()
results_df.to_csv("evaluation2_spbe_vector.csv", index=False)
print(results_df)

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

📊 Hasil Evaluasi RAGAS:
                                          user_input  \
0  Dalam penerapan Manajemen Risiko SPBE, masing-...   
1  Jika "Rencana dan Anggaran SPBE Instansi Pusat...   
2  Berapa nilai indeks domain yang masih dibawah ...   
3  Sebuah IPPD melampirkan Peta Rencana SPBE yang...   
4  Dibawah ini merupakan domain-domain dari Arsit...   

                                  retrieved_contexts  \
0  [INDIKATOR 21\nDomain Aspek Ind Kuesioner D3 A...   
1  [d. Tidak menerima gratifikasi terkait dengan ...   
2  [Berdasarkan capaian Indeks SPBE Nasional seba...   
3  [d. Tidak menerima gratifikasi terkait dengan ...   
4  [jdih.menpan.go.id\n50-\nd. Arsitektur SPBE Pe...   

                                            response  \
0  Dalam penerapan Manajemen Risiko SPBE, masing-...   
1  Berdasarkan pedoman pemantauan dan evaluasi SP...   
2  Nilai indeks domain yang masih di bawah target...   
3  Berdasarkan pedoman pemantauan dan evaluasi SP...   
4  Domain-domain dari 

In [58]:
results_df

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_recall,context_precision
0,"Dalam penerapan Manajemen Risiko SPBE, masing-...",[INDIKATOR 21\nDomain Aspek Ind Kuesioner D3 A...,"Dalam penerapan Manajemen Risiko SPBE, masing-...",PermenPANRB No. 5 Tahun 2020 memberikan pedoma...,0.500000,0.936441,1.000000,0.500000
1,"Jika ""Rencana dan Anggaran SPBE Instansi Pusat...",[d. Tidak menerima gratifikasi terkait dengan ...,Berdasarkan pedoman pemantauan dan evaluasi SP...,Berdasarkan pedoman pemantauan dan evaluasi SP...,0.777778,0.678468,0.142857,0.469286
2,Berapa nilai indeks domain yang masih dibawah ...,[Berdasarkan capaian Indeks SPBE Nasional seba...,Nilai indeks domain yang masih di bawah target...,Nilai indeks domain yang masih di bawah target...,1.000000,0.748882,1.000000,0.500000
3,Sebuah IPPD melampirkan Peta Rencana SPBE yang...,[d. Tidak menerima gratifikasi terkait dengan ...,Berdasarkan pedoman pemantauan dan evaluasi SP...,Berdasarkan pedoman pemantauan dan evaluasi SP...,0.600000,0.762038,1.000000,0.444577
4,Dibawah ini merupakan domain-domain dari Arsit...,[jdih.menpan.go.id\n50-\nd. Arsitektur SPBE Pe...,Domain-domain dari Arsitektur SPBE berdasarkan...,Domain Arsitektur Manajemen SPBE tidak termasu...,0.750000,0.939570,1.000000,1.000000


In [ ]:
# results_df.to_csv("results_spbe_vector_ragas.csv")